# MotifForge: text to SVG with a small Qwen3 model

Train a Qwen3 text model to turn a description into a clean SVG icon or textile motif.

1. Kaggle: Settings > Accelerator > GPU T4 x2, and Internet on.
2. Edit REPO_URL below to your fork.
3. Run the cells in order.
4. Download the evidence ZIP and the Q4_K_M GGUF at the end.

The kit lives in `motifforge/` inside the repository.

In [ ]:
!nvidia-smi

## 1. Install training and rendering packages

In [ ]:
%pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer pyyaml
%pip install -q torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
%pip install -q --no-deps unsloth_zoo bitsandbytes accelerate peft triton unsloth
!unsloth install-kernels
%pip install -q --no-deps --upgrade "torchao>=0.16.0"
%pip install -q transformers==4.57.6
%pip install -q --no-deps trl==0.22.2
%pip install -q cairosvg scikit-image pillow
!sudo apt-get -qq update && sudo apt-get -qq install -y libcairo2 || true

## 2. Settings, repository and data

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata
import json
import platform
import subprocess
import sys
import time
import yaml

REPO_URL = "https://github.com/YOUR-GITHUB-USER/finetune-anywhere.git"
REPO_DIR = Path("/kaggle/working/finetune-anywhere")
if not (REPO_DIR / "motifforge").exists():
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
ROOT = REPO_DIR
sys.path.insert(0, str(ROOT / "motifforge"))
sys.path.insert(0, str(ROOT / "scripts"))

DATA_MODE = "prepare"
N_TRAIN = 6000
N_VALID = 400
N_EVAL = 60
MAX_SVG_TOKENS = 1800

config = yaml.safe_load((ROOT / "motifforge/config.yaml").read_text())
MODEL_ALIAS = config["model_alias"]
registry = yaml.safe_load((ROOT / "configs/models.yaml").read_text())
model_entry = registry["models"][MODEL_ALIAS]

RUN_NAME = "motifforge-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUTPUT = Path("/kaggle/working/runs") / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)
DATA_DIR = OUTPUT / "data"
if DATA_MODE == "prepare":
    subprocess.run([sys.executable, str(ROOT / "motifforge/prepare_data.py"),
        "--out", str(DATA_DIR),
        "--n-train", str(N_TRAIN), "--n-valid", str(N_VALID), "--n-eval", str(N_EVAL),
        "--max-svg-tokens", str(MAX_SVG_TOKENS), "--tokenizer", MODEL_ALIAS], check=True)
else:
    source = Path("/kaggle/input/motifforge-data")
    DATA_DIR.mkdir(exist_ok=True)
    for split in ["train", "valid", "eval"]:
        target = DATA_DIR / (split + ".jsonl")
        if not target.exists():
            target.symlink_to(source / (split + ".jsonl"))
print("Model:", model_entry["id"])
print("Data:", DATA_DIR)
print("Output:", OUTPUT)

## 3. Load the base model in 4-bit

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

assert torch.cuda.is_available(), "Select a GPU T4 x2 runtime"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_entry["id"],
    max_seq_length=config["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

## 4. Apply LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=config["rank"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=config["alpha"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=config["seed"],
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()

## 5. Fix Qwen3 non-thinking mode in the chat template

In [ ]:
from configure_chat import configure_tokenizer

configure_tokenizer(tokenizer)
preview = [
    {"role": "user", "content": "A minimal three petal flower motif, two colors, centered."},
    {"role": "assistant", "content": '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 64 64"><circle cx="32" cy="32" r="24" fill="#1f6f78"/></svg>'},
]
print(tokenizer.apply_chat_template(preview, tokenize=False, add_generation_prompt=False, enable_thinking=False))

## 6. Load and validate the chat data

In [ ]:
from datasets import Dataset
from common import read_rows
from validate_dataset import validate

raw = {}
seen = set()
for split in ["train", "valid", "eval"]:
    path = DATA_DIR / (split + ".jsonl")
    errors, lengths, prompts = validate(path, tokenizer=tokenizer, max_tokens=config["max_seq_length"])
    if errors:
        raise ValueError("\n".join(errors))
    if seen & prompts:
        raise ValueError(split + ": a prompt appears in another split")
    seen |= prompts
    raw[split] = read_rows(path)
    print(split, len(raw[split]), "examples; longest tokens:", max(lengths))

def render_chat(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False,
            add_generation_prompt=False, enable_thinking=False)}

train_data = Dataset.from_list(raw["train"]).map(render_chat, remove_columns=["messages"])
valid_data = Dataset.from_list(raw["valid"]).map(render_chat, remove_columns=["messages"])
print(train_data[0]["text"][:600])

## 7. Record base answers on held-out prompts

In [ ]:
GENERATION = {"max_new_tokens": config["max_new_tokens"], "do_sample": False}

def answer(messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False,
        add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, **GENERATION, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

FastLanguageModel.for_inference(model)
base_outputs = []
with model.disable_adapter():
    for index, row in enumerate(raw["eval"], 1):
        base_outputs.append(answer(row["messages"][:-1]))
        if index % 10 == 0:
            print("base", index, "of", len(raw["eval"]))
print("PROMPT:", raw["eval"][0]["messages"][-2]["content"])
print("BASE:", base_outputs[0][:400])
FastLanguageModel.for_training(model)

## 8. Train

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_data,
    eval_dataset=valid_data,
    args=SFTConfig(
        output_dir=str(OUTPUT / "checkpoints"),
        dataset_text_field="text",
        max_length=config["max_seq_length"],
        per_device_train_batch_size=config["batch_size"],
        gradient_accumulation_steps=config["gradient_accumulation_steps"],
        learning_rate=config["learning_rate"],
        num_train_epochs=config["epochs"],
        warmup_ratio=0.1,
        optim="adamw_8bit",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        per_device_eval_batch_size=1,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        logging_steps=10,
        seed=config["seed"],
        report_to="none",
        packing=False,
        padding_free=False,
    ),
)
FastLanguageModel.for_training(model)
torch.cuda.reset_peak_memory_stats()
started = time.monotonic()
result = trainer.train()
train_seconds = time.monotonic() - started
print("Train seconds:", round(train_seconds, 2))
print("Peak VRAM GiB:", round(torch.cuda.max_memory_allocated() / 2**30, 2))

## 9. Generate fine-tuned answers and pair the records

In [ ]:
FastLanguageModel.for_inference(model)
tuned_outputs = []
for index, row in enumerate(raw["eval"], 1):
    tuned_outputs.append(answer(row["messages"][:-1]))
    if index % 10 == 0:
        print("tuned", index, "of", len(raw["eval"]))

records = [
    {
        "prompt": row["messages"][-2]["content"],
        "reference_svg": row["messages"][-1]["content"],
        "base_svg": base,
        "tuned_svg": tuned,
    }
    for row, base, tuned in zip(raw["eval"], base_outputs, tuned_outputs)
]
print("First tuned preview:", tuned_outputs[0][:400])

## 10. Score valid, render and SSIM before export

In [ ]:
from svg_utils import extract_svg, render_svg, ssim_score, ink_ratio, validate_svg

def score(svg, reference, size=256):
    clean = extract_svg(svg) or (svg if isinstance(svg, str) else "")
    row = {"valid": False, "reason": "empty", "render": False, "ssim": None, "ink": None}
    if not clean.strip():
        return row
    ok, reason = validate_svg(clean)
    row["valid"], row["reason"] = bool(ok), reason
    if not ok:
        return row
    image = render_svg(clean, size=size)
    if image is None:
        row["reason"] = "render failed"
        return row
    row["render"] = True
    row["ink"] = round(ink_ratio(image), 4)
    ref_clean = extract_svg(reference) or reference
    ref_image = render_svg(ref_clean, size=size) if ref_clean else None
    if ref_image is not None:
        row["ssim"] = round(ssim_score(image, ref_image), 4)
    return row

scored = [
    {
        "prompt": record["prompt"],
        "base": score(record["base_svg"], record["reference_svg"]),
        "tuned": score(record["tuned_svg"], record["reference_svg"]),
    }
    for record in records
]

def summarize(name):
    rows = [row[name] for row in scored]
    n = len(rows)
    ssims = [row["ssim"] for row in rows if row["ssim"] is not None]
    return {
        "n": n,
        "valid_rate": round(sum(row["valid"] for row in rows) / n, 3),
        "render_rate": round(sum(row["render"] for row in rows) / n, 3),
        "blank_rate": round(sum(1 for row in rows if row["render"] and row["ink"] < 0.001) / n, 3),
        "mean_ssim": round(sum(ssims) / len(ssims), 4) if ssims else None,
        "ssim_pairs": len(ssims),
    }

summary = {"base": summarize("base"), "tuned": summarize("tuned")}
print(json.dumps(summary, indent=2))
(OUTPUT / "score_summary.json").write_text(json.dumps(summary, indent=2) + "\n")

## 11. Save the adapter and evidence

In [ ]:
model.save_pretrained(str(OUTPUT / "adapter"))
tokenizer.save_pretrained(str(OUTPUT / "adapter"))
(OUTPUT / "comparison.json").write_text(json.dumps(records, indent=2, ensure_ascii=False) + "\n")
(OUTPUT / "scored.json").write_text(json.dumps(scored, indent=2, ensure_ascii=False) + "\n")

def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

report = {
    "project": "motifforge",
    "recipe": config["recipe"],
    "model_alias": MODEL_ALIAS,
    "model_id": model_entry["id"],
    "catalog_revision": model_entry["revision"],
    "repository_commit": subprocess.check_output(
        ["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True
    ).strip(),
    "gpu": torch.cuda.get_device_name(0),
    "python": platform.python_version(),
    "packages": {
        name: version(name)
        for name in ["unsloth", "torch", "transformers", "trl", "peft", "datasets", "cairosvg"]
    },
    "settings": config,
    "generation": GENERATION,
    "train_seconds": train_seconds,
    "train_metrics": result.metrics,
    "peak_vram_bytes": torch.cuda.max_memory_allocated(),
    "split_sizes": {key: len(value) for key, value in raw.items()},
    "score_summary": summary,
    "export_completed": False,
}
(OUTPUT / "run.json").write_text(json.dumps(report, indent=2) + "\n")
print("Saved adapter and evidence to", OUTPUT)

## 12. Export a Q4_K_M GGUF

In [ ]:
export_started = time.monotonic()
model.save_pretrained_gguf(str(OUTPUT / "gguf"), tokenizer, quantization_method="q4_k_m")
ggufs = sorted(
    path for path in OUTPUT.rglob("*.gguf")
    if path.is_file() and path.stat().st_size > 0 and "Q4_K_M" in path.name.upper()
)
assert ggufs, "No Q4_K_M GGUF found; read the export log above"
report["export_completed"] = True
report["export_seconds"] = time.monotonic() - export_started
report["gguf_files"] = [
    {"name": path.name, "path": str(path.relative_to(OUTPUT)), "bytes": path.stat().st_size}
    for path in ggufs
]
(OUTPUT / "run.json").write_text(json.dumps(report, indent=2) + "\n")
for path in ggufs:
    print(path, round(path.stat().st_size / 1e9, 2), "GB")

## 13. Download evidence and model

In [ ]:
import zipfile
bundle = OUTPUT / "motifforge-evidence.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ["run.json", "score_summary.json", "comparison.json", "scored.json"]:
        path = OUTPUT / name
        if path.exists():
            archive.write(path, arcname=name)
print("Evidence:", bundle)
try:
    from IPython.display import FileLink, display
    display(FileLink(str(bundle)))
    for path in sorted(OUTPUT.rglob("*.gguf")):
        display(FileLink(str(path)))
except Exception as exc:
    print("Open the outputs panel to download files:", exc)

## 14. Next on your laptop

1. Download the Q4_K_M GGUF.
2. Put it next to `motifforge/Modelfile` and run `ollama create motifforge -f Modelfile`.
3. Run `python motifforge/app.py --model motifforge` for the demo.
4. Run `python motifforge/evaluate.py --eval-jsonl motifforge/eval_prompts_motifs.jsonl --base-model qwen3:1.7b --tuned-model motifforge --out outputs/motifforge-local --limit 20`.

See `motifforge/README.md` for the full checklist.